# اليوم 3، المعمل 5: الأشجار التجميعية

نقارن Random Forest مع النموذج الخطي تحت نفس بروتوكول التقييم. النموذج الأقوى لا يفوز إلا إذا كان التحسن ثابتًا وله تكلفة مقبولة.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "data" / "raw"
RANDOM_STATE = 42


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from manafeth.data import load_customers, split_customers
from manafeth.features import build_preprocessor
from manafeth.evaluation import evaluate_classifier

df = load_customers(DATA)
X_train, X_test, y_train, y_test = split_customers(df)
forest = Pipeline([("prep", build_preprocessor(scale_numeric=False)), ("model", RandomForestClassifier(n_estimators=250, max_depth=10, min_samples_leaf=15, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE))])


## مهمتك

قيّم forest باستخدام الدالة المشتركة. قارن PR-AUC ووقت التدريب بالنموذج الخطي من المعمل السابق.

In [ ]:
score = evaluate_classifier(forest, X_train, y_train)
display(score.to_frame("random_forest").T)

from sklearn.model_selection import train_test_split
X_fit, X_valid, y_fit, y_valid = train_test_split(X_train, y_train, test_size=.20, stratify=y_train, random_state=RANDOM_STATE)
forest.fit(X_fit, y_fit)
importance = permutation_importance(forest, X_valid, y_valid, scoring="average_precision", n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.Series(importance.importances_mean, index=X_valid.columns).sort_values(ascending=False).head(10)
imp.sort_values().plot.barh(title="Permutation importance on validation data")
plt.xlabel("Decrease in PR-AUC"); plt.show()


## Permutation Importance

درّب على جزء train واحسب الأهمية على validation الخام. هذا الاختبار يسأل: كم تنخفض النتيجة عندما نفسد عمودًا واحدًا؟

In [ ]:
# أُنجزت خطوات هذا القسم في خلية الحل السابقة.


**ناتج التسليم:** `CHAMPION.md` يحتوي جدول النماذج وسبب الاختيار وسبب الاحتفاظ بالمنافس.